# BFloat16 Training Tutorial

## Overview

BF16 (Brain Floating Point) provides the dynamic range of FP32 with reduced precision, eliminating the need for loss scaling.

### Learning Objectives
- Understand BF16 vs FP16 differences
- Implement BF16 training without loss scaling
- Know hardware requirements

### References
- Kalamkar et al., "A Study of BFLOAT16 for Deep Learning Training", 2019

## 1. BF16 vs FP16 Comparison

| Property | FP16 | BF16 | FP32 |
|----------|------|------|------|
| Sign bits | 1 | 1 | 1 |
| Exponent bits | 5 | 8 | 8 |
| Mantissa bits | 10 | 7 | 23 |
| Max value | 65,504 | 3.4e38 | 3.4e38 |
| Min positive | 6.1e-5 | 1.2e-38 | 1.2e-38 |
| Loss scaling | Required | Not needed | N/A |

**Key Insight**: BF16 has same exponent range as FP32, so gradients rarely underflow.

In [ ]:
import torch
import torch.nn as nn

# Check BF16 support
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    capability = torch.cuda.get_device_capability()
    bf16_supported = capability[0] >= 8  # Ampere (SM80) or newer
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Compute Capability: {capability[0]}.{capability[1]}")
    print(f"BF16 Supported: {bf16_supported}")

## 2. BF16 Training Implementation

In [ ]:
def train_with_bf16(model, dataloader, optimizer, criterion, device, epochs=5):
    """Training with BF16 - no loss scaling needed!"""
    model.to(device)
    
    for epoch in range(epochs):
        total_loss = 0
        for data, target in dataloader:
            data = data.to(device, dtype=torch.bfloat16)
            target = target.to(device)
            
            optimizer.zero_grad()
            
            # Forward with autocast to BF16
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                output = model(data)
                loss = criterion(output, target)
            
            # No scaling needed - direct backward
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

## 3. Summary

### When to Use BF16

| Scenario | Recommendation |
|----------|----------------|
| Ampere+ GPU (A100, H100) | Use BF16 |
| Older GPUs (V100, etc.) | Use FP16 + scaling |
| Training stability issues | Try BF16 |
| Maximum precision needed | Use FP32 |